1. Kéo bên trái và đọc file json cấp xã "gadm41_VNM_3.json" hiện thị thông tin Cấp Xã / Phường / Thị trấn

In [ ]:
from collections import defaultdict
import json

# Đường dẫn tới file JSON trên máy của bạn
file_path = "/content/gadm41_VNM_3.json"

print("Đang đọc và phân tích dữ liệu...")

# Cấu trúc lưu trữ: {tên_tỉnh: {loại_đơn_vị: số_lượng}}
stats = defaultdict(lambda: defaultdict(int))
total_units = 0

with open(file_path, "r", encoding="utf-8") as f:
  data = json.load(f)

for feature in data.get("features", []):
  props = feature.get("properties", {})
  province = props.get("NAME_1", "Không xác định")
  unit_type = props.get("TYPE_3", "Khác")

  stats[province][unit_type] += 1
  total_units += 1

# --- XUẤT KẾT QUẢ THỐNG KÊ ---
provinces = sorted(stats.keys())
print("\n" + "=" * 60)
print(f"TỔNG KẾT TOÀN DIỆN:")
print(f"- Tổng số tỉnh/thành phố: {len(provinces)}")
print(f"- Tổng số đơn vị cấp 3 (xã/phường/thị trấn...): {total_units}")
print("=" * 60)

print(
    f"{'STT':<5} | {'Tỉnh / Thành phố':<25} | {'Tổng cấp 3':<12} | Chi tiết phân loại"
)
print("-" * 75)

for idx, prov in enumerate(provinces, start=1):
  prov_total = sum(stats[prov].values())
  breakdown = ", ".join([f"{k}: {v}" for k, v in stats[prov].items()])
  print(f"{idx:<5} | {prov:<25} | {prov_total:<12} | {breakdown}")

2. Tự động đọc dữ liệu geojson từ nguồn online --> tên tỉnh --> huyện --> xã (vị trí địa lý Geometry) --> lưu vào file vietnam_communes_epsg4326.geojson
- Kiểm tra & Chuẩn hóa Hệ quy chiếu (CRS) về EPSG:4326

In [ ]:
import os
import zipfile
import requests
import geopandas as gpd
from shapely.validation import make_valid

# -------------------------------------------------------------
# 1. Cấu hình đường dẫn & URL GADM 4.1 cho Việt Nam
# -------------------------------------------------------------
DATA_DIR = "./data_spatial"
ZIP_PATH = os.path.join(DATA_DIR, "gadm41_VNM_shp.zip")
EXTRACT_DIR = os.path.join(DATA_DIR, "gadm41_VNM")
OUTPUT_CLEAN_PATH = os.path.join(DATA_DIR, "vietnam_communes_epsg4326.geojson")

os.makedirs(DATA_DIR, exist_ok=True)

# URL chính thức của GADM 4.1 cho gói Shapefile Việt Nam
GADM_VNM_URL = "https://geodata.ucdavis.edu/gadm/gadm4.1/shp/gadm41_VNM_shp.zip"

# -------------------------------------------------------------
# 2. Tải và giải nén dữ liệu
# -------------------------------------------------------------
if not os.path.exists(ZIP_PATH):
    print(f"[*] Đang tải dữ liệu GADM từ: {GADM_VNM_URL}...")
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(GADM_VNM_URL, headers=headers, stream=True)
    response.raise_for_status()
    with open(ZIP_PATH, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)
    print("[+] Tải thành công.")
else:
    print("[*] File nén đã tồn tại, bỏ qua bước tải.")

if not os.path.exists(EXTRACT_DIR):
    print("[*] Đang giải nén file...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print("[+] Giải nén hoàn tất.")

# -------------------------------------------------------------
# 3. Đọc dữ liệu Level 3 (Cấp Xã/Phường/Thị trấn)
# -------------------------------------------------------------
shp_level3_path = os.path.join(EXTRACT_DIR, "gadm41_VNM_3.shp")
print(f"[*] Đang nạp Shapefile cấp xã: {shp_level3_path}")
gdf_communes = gpd.read_file(shp_level3_path)

# -------------------------------------------------------------
# 4. Kiểm tra & Chuẩn hóa Hệ quy chiếu (CRS) về EPSG:4326
# -------------------------------------------------------------
target_crs = "EPSG:4326"

print(f"[*] CRS ban đầu: {gdf_communes.crs}")
if gdf_communes.crs is None:
    print("[!] Dữ liệu không có thông tin CRS, thiết lập mặc định EPSG:4326.")
    gdf_communes = gdf_communes.set_crs(target_crs)
elif gdf_communes.crs.to_string() != target_crs:
    print(f"[*] Chuyển đổi CRS từ {gdf_communes.crs} -> {target_crs}...")
    gdf_communes = gdf_communes.to_crs(target_crs)
else:
    print("[+] Dữ liệu đã ở định dạng chuẩn WGS 84 (EPSG:4326).")

# -------------------------------------------------------------
# 5. Sửa lỗi đa giác hình học (Geometry Cleaning & Validation)
# -------------------------------------------------------------
# Dữ liệu viễn thám dễ bị crash nếu đa giác xã bị tự cắt (self-intersection)
invalid_count = (~gdf_communes.is_valid).sum()
if invalid_count > 0:
    print(f"[!] Phát hiện {invalid_count} đa giác xã bị lỗi hình học. Đang tự động sửa...")
    gdf_communes["geometry"] = gdf_communes["geometry"].apply(make_valid)

# -------------------------------------------------------------
# 6. Kiểm tra cấu trúc thuộc tính và lưu file sạch
# -------------------------------------------------------------
cols_to_keep = ["GID_1", "NAME_1", "GID_2", "NAME_2", "GID_3", "NAME_3", "TYPE_3", "geometry"]
gdf_cleaned = gdf_communes[cols_to_keep].copy()

# Xuất ra GeoJSON chuẩn để dễ nạp lên GEE Asset hoặc chạy trên PyTorch
gdf_cleaned.to_file(OUTPUT_CLEAN_PATH, driver="GeoJSON")

print("\n--- KẾT QUẢ KIỂM TRA DỮ LIỆU ---")
print(f"- Tổng số đơn vị cấp xã: {len(gdf_cleaned):,}")
print(f"- Hệ tọa độ cuối cùng: {gdf_cleaned.crs}")
print(f"- {len(gdf_cleaned.columns)} cột định danh: {list(gdf_cleaned.columns[:-1])}")
print(f"- File lưu tại: {OUTPUT_CLEAN_PATH}")

In [ ]:
print(gdf_cleaned.shape)
print(gdf_cleaned.head(5))
print(gdf_cleaned[gdf_cleaned["GID_1"]== "VNM.27_1"])

2. KẾT NỐI GOOLE EARTH ENGINE

In [ ]:
import ee

# 1. Kích hoạt xác thực (nếu đã xác thực rồi thì cell sẽ tự bỏ qua)
try:
    ee.Initialize(project="angelic-digit-434804-r9")
    print("[+] GEE đã được khởi tạo thành công!")
except Exception as e:
    print("[*] Yêu cầu xác thực tài khoản Google...")
    ee.Authenticate()
    ee.Initialize(project="angelic-digit-434804-r9")
    print("[+] Xác thực và khởi tạo GEE hoàn tất!")

3. Đọc features Spectral_Indices (band màu) của ảnh ngày cấp xã và lưu vào drive "/content"

In [ ]:
import calendar
import os
import re
import unicodedata
import ee
import pandas as pd

PROJECT_ID = "angelic-digit-434804-r9"
ASSET_ID = f"projects/{PROJECT_ID}/assets/vietnam_communes_l3"

ee.Initialize(project=PROJECT_ID)
communes_fc = ee.FeatureCollection(ASSET_ID)


def normalize_str(s):
  if not s:
    return ""
  s = unicodedata.normalize("NFD", str(s))
  s = re.sub(r"[\u0300-\u036f]", "", s)
  s = s.replace("đ", "d").replace("Đ", "d")
  return re.sub(r"[^a-zA-Z0-9]", "", s).lower()


# 1. Hàm lọc mây và chuẩn hóa tên kênh
def mask_s2_sr(img):
  qa = img.select("QA60")
  cloud_mask = (qa.bitwiseAnd(1 << 10).eq(0)).And(qa.bitwiseAnd(1 << 11).eq(0))
  return (
      img.updateMask(cloud_mask)
      .divide(10000)
      .select(
          ["B2", "B3", "B4", "B8", "B11", "B12"],
          ["BLUE", "GREEN", "RED", "NIR", "SWIR1", "SWIR2"],
      )
  )


# 2. Hàm tính 4 chỉ số phổ
def add_indices(img):
  ndvi = img.normalizedDifference(["NIR", "RED"]).rename("NDVI")
  ndbi = img.normalizedDifference(["SWIR1", "NIR"]).rename("NDBI")
  mndwi = img.normalizedDifference(["GREEN", "SWIR1"]).rename("MNDWI")
  bsi = img.expression(
      "((SWIR1 + RED) - (NIR + BLUE)) / ((SWIR1 + RED) + (NIR + BLUE))",
      {
          "SWIR1": img.select("SWIR1"),
          "RED": img.select("RED"),
          "NIR": img.select("NIR"),
          "BLUE": img.select("BLUE"),
      },
  ).rename("BSI")
  return img.addBands([ndvi, ndbi, mndwi, bsi])


# 3. Hàm trích xuất dữ liệu và xuất file CSV kèm hậu tố "Spectral_Indices"
def export_province_s2_local(
    province_gid,
    province_name,
    year,
    month,
    save_dir="/content",
    gdf_admin=None,
):
  prov_prefix = province_gid.replace("_1", "") + "."
  prov_communes = communes_fc.filter(
      ee.Filter.stringStartsWith("GID_3", prov_prefix)
  )

  _, last_day = calendar.monthrange(year, month)
  s_date = f"{year}-{month:02d}-01"
  e_date = f"{year}-{month:02d}-{last_day:02d}"

  collection_id = (
      "COPERNICUS/S2_SR_HARMONIZED" if year >= 2019 else "COPERNICUS/S2_HARMONIZED"
  )
  print(f"[*] Nguồn dữ liệu: {collection_id}")
  print(f"[*] Tìm kiếm ảnh từ ngày {s_date} đến {e_date}...")

  raw_col = (
      ee.ImageCollection(collection_id)
      .filterBounds(prov_communes)
      .filterDate(s_date, e_date)
  )

  img_count = raw_col.size().getInfo()
  print(f"[*] Tìm thấy {img_count} cảnh ảnh Sentinel-2 trong phạm vi tỉnh.")

  if img_count == 0:
    print(
        f"[-] LỖI: Không có cảnh ảnh nào trong tháng {month:02d}/{year} cho tỉnh"
        f" {province_name}."
    )
    return None

  filtered_col = raw_col.filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 85))
  if filtered_col.size().getInfo() == 0:
    print(
        "[!] Cảnh báo: Tháng này nhiều mây (>85%), giữ lại ảnh gốc để tổng"
        " hợp..."
    )
    composite = raw_col.map(mask_s2_sr).median()
  else:
    composite = filtered_col.map(mask_s2_sr).median()

  tensor = add_indices(composite)
  bands = [
      "BLUE",
      "GREEN",
      "RED",
      "NIR",
      "SWIR1",
      "SWIR2",
      "NDVI",
      "NDBI",
      "MNDWI",
      "BSI",
  ]
  reducers = ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True)

  print(f"[*] Đang thực hiện thống kê vùng (ReduceRegions) cấp xã...")
  stats = tensor.select(bands).reduceRegions(
      collection=prov_communes,
      reducer=reducers,
      scale=50,
      tileScale=4,
      crs="EPSG:4326",
  )

  feature_cols = [
      "BLUE_mean",
      "BLUE_stdDev",
      "GREEN_mean",
      "GREEN_stdDev",
      "RED_mean",
      "RED_stdDev",
      "NIR_mean",
      "NIR_stdDev",
      "SWIR1_mean",
      "SWIR1_stdDev",
      "SWIR2_mean",
      "SWIR2_stdDev",
      "NDVI_mean",
      "NDVI_stdDev",
      "NDBI_mean",
      "NDBI_stdDev",
      "MNDWI_mean",
      "MNDWI_stdDev",
      "BSI_mean",
      "BSI_stdDev",
  ]
  selected_cols = ["GID_3"] + feature_cols

  print(f"[*] Đang tải dữ liệu kết quả từ GEE về máy...")
  features = stats.select(selected_cols).getInfo()["features"]
  rows = [f["properties"] for f in features]
  df_s2 = pd.DataFrame(rows)

  if gdf_admin is not None:
    admin_cols = [
        "GID_1",
        "NAME_1",
        "GID_2",
        "NAME_2",
        "GID_3",
        "NAME_3",
        "TYPE_3",
    ]
    existing_cols = [c for c in admin_cols if c in gdf_admin.columns]
    lookup = gdf_admin[existing_cols].drop_duplicates(subset=["GID_3"])
    df_res = pd.merge(lookup, df_s2, on="GID_3", how="inner")
  else:
    df_res = df_s2

  # Đặt tên file: s2_[tên_tỉnh]_[mã_tỉnh]_[năm]_[tháng]_Spectral_Indices.csv
  clean_name = normalize_str(province_name)
  safe_gid = province_gid.replace("/", "_")
  out_filename = (
      f"s2_{clean_name}_{safe_gid}_{year}_{month:02d}_Spectral_Indices.csv"
  )
  out_path = os.path.join(save_dir, out_filename)

  df_res.to_csv(out_path, index=False, encoding="utf-8-sig")
  print(f"\n[+] XUẤT FILE THÀNH CÔNG: {out_path}")
  print(f"[+] Đã ghi nhận {len(df_res)} xã.")
  return df_res


# 4. Trích xuất danh mục Tỉnh
prov_map = {}
if "gdf_cleaned" in globals() and "NAME_1" in gdf_cleaned.columns:
  print("[*] Đọc danh mục tỉnh từ 'gdf_cleaned'...")
  unique_provs = (
      gdf_cleaned[["NAME_1", "GID_1"]].drop_duplicates().dropna().values
  )
  for name, gid1 in unique_provs:
    prov_map[str(name)] = str(gid1)
else:
  print("[*] Đọc danh mục tỉnh từ Asset GEE...")
  first_feature = communes_fc.first().getInfo()["properties"]
  col_name = "NAME_1" if "NAME_1" in first_feature else None
  if col_name:
    data = communes_fc.reduceColumns(
        ee.Reducer.toList(2), [col_name, "GID_3"]
    ).getInfo()["list"]
    for name, gid3 in data:
      if name and gid3:
        parts = str(gid3).split(".")
        if len(parts) >= 2:
          gid1 = f"{parts[0]}.{parts[1]}_1"
          prov_map[str(name)] = gid1

if not prov_map:
  gid3_list = communes_fc.aggregate_array("GID_3").getInfo()
  distinct_gids = sorted(
      list(
          set(
              [
                  f"{x.split('.')[0]}.{x.split('.')[1]}_1"
                  for x in gid3_list
                  if "." in str(x)
              ]
          )
      )
  )
  for gid in distinct_gids:
    prov_map[gid] = gid

# 5. Hiển thị danh sách tỉnh kèm mã GID
print("=" * 88)
print(f"[+] DANH MỤC {len(prov_map)} TỈNH/THÀNH PHỐ KÈM MÃ PROVINCE:")
items = [f"{name} ({gid})" for name, gid in sorted(prov_map.items())]
for i in range(0, len(items), 2):
  row_items = items[i : i + 2]
  print("  ".join(f"{item:<42}" for item in row_items))
print("=" * 88)

# 6. Người dùng nhập tham số
user_prov = input("\n👉 1. Nhập TÊN hoặc MÃ tỉnh/thành phố: ").strip()
norm_prov = normalize_str(user_prov)

matched_name = None
for official_name, gid in prov_map.items():
  if (
      normalize_str(official_name) == norm_prov
      or normalize_str(gid) == norm_prov
  ):
    matched_name = official_name
    break

if not matched_name:
  print(f"\n[-] LỖI: Không tìm thấy tỉnh nào khớp với '{user_prov}'.")
else:
  target_gid = prov_map[matched_name]
  print(f"[+] Xác nhận tỉnh: '{matched_name}' (Mã: {target_gid})")

  while True:
    year_input = input("👉 2. Nhập NĂM (2016 - 2026): ").strip()
    if year_input.isdigit() and 2016 <= int(year_input) <= 2026:
      selected_year = int(year_input)
      break
    print("[-] Năm không hợp lệ. Vui lòng nhập lại!")

  while True:
    month_input = input("👉 3. Nhập THÁNG (1 - 12): ").strip()
    if month_input.isdigit() and 1 <= int(month_input) <= 12:
      selected_month = int(month_input)
      break
    print("[-] Tháng phải từ 1 đến 12. Vui lòng nhập lại!")

  print("\n" + "=" * 65)
  print(
      f"[*] BẮT ĐẦU XỬ LÝ: {matched_name} ({target_gid}) - Tháng"
      f" {selected_month:02d}/{selected_year}"
  )
  print("=" * 65)

  df_result = export_province_s2_local(
      province_gid=target_gid,
      province_name=matched_name,
      year=selected_year,
      month=selected_month,
      save_dir="/content",
      gdf_admin=gdf_cleaned if "gdf_cleaned" in globals() else None,
  )

  if df_result is not None:
    display(df_result.head(10))

Với thông số kỹ thuật cụ thể của tệp ảnh:

* **Độ phân giải không gian (Spatial Resolution):** $\sim 19.3\text{m} \times 19.3\text{m}$ (mỗi pixel tương đương diện tích mặt đất khoảng $\approx 372\text{ m}^2$).
* **Kích thước khung hình:** $157 \times 147$ pixels (tổng diện tích bao phủ khoảng $3\text{ km} \times 2.8\text{ km} \approx 8.5\text{ km}^2$, trong đó diện tích thực của xã khoảng $4.07\text{ km}^2$).
* **Độ sâu quang phổ:** **10 kênh phổ (Bands)** gồm đầy đủ dải ánh sáng nhìn thấy (RGB), Cận hồng ngoại (NIR), và Hồng ngoại sóng ngắn (SWIR1, SWIR2).

Khả năng nhận diện đối tượng (Object Detection / Semantic Segmentation) được phân định cụ thể theo kích thước hình học và đặc tính phản xạ phổ như sau:

---

### 1. Các đối tượng **DETECT ĐƯỢC RẤT RÕ NÉT VÀ CHUẨN XÁC**

Đây là những thực thể nhân tạo hoặc tự nhiên có kích thước bề mặt tối thiểu từ **vài trăm mét vuông trở lên** (chiếm từ $2 - 3$ pixel liền kề):

* **Khu công nghiệp, nhà máy, kho bãi logistics quy mô lớn:**
* Các nhà xưởng thường có diện tích từ $1.000\text{ m}^2$ đến hàng chục nghìn $\text{m}^2$ (tương đương $5 - 50+$ pixel).
* **Kênh nhận diện:** Kênh **SWIR1 và SWIR2** cực kỳ nhạy với vật liệu kim loại (mái tôn lợp xưởng) và bề mặt bê tông công nghiệp, tạo nên tín hiệu phản xạ trắng/sáng rõ rệt tách biệt hoàn toàn khỏi khu dân cư thông thường.


* **Mạng lưới giao thông trục chính:**
* Đường cao tốc, quốc lộ, đại lộ đô thị hoặc đường liên huyện có lộ giới mặt đường $> 15 - 20\text{m}$ (chiếm bề ngang khoảng $1$ pixel và kéo dài liên tục thành vệt).
* Nút giao thông khác mức, cầu vượt sông lớn, cảng bốc dỡ ven sông.


* **Mặt nước tự nhiên và hệ thống nuôi trồng thủy sản:**
* Sông chính, kênh rạch lớn (rộng $> 20\text{m}$), hồ thủy lợi, hồ chứa nước đô thị.
* Hệ thống ao/đầm nuôi tôm cá dạng ô bàn cờ (mỗi ô thường $> 500 - 1.000\text{m}^2$). Sự kết hợp giữa kênh **Green và SWIR (chỉ số MNDWI)** cho phép bóc tách ranh giới bờ đập và mặt nước rất sắc.


* **Khối công trình công cộng & hạ tầng đô thị tập trung:**
* Bệnh viện tuyến huyện/tỉnh, trường học cấp 3/đại học (khối nhà nhiều tầng kèm sân tập), sân vận động, trung tâm hành chính xã/huyện.
* Các cụm chung cư, khối nhà cao tầng khu đô thị mới.


* **Mảng cấu trúc nông nghiệp & tự nhiên quy mô lớn:**
* Cánh đồng lúa tập trung, vùng chuyên canh hoa màu, trang trại trồng cây ăn trái/cây công nghiệp (cao su, cà phê).
* Đồi rừng, công viên tập trung, dải cây xanh ven sông (nhờ tín hiệu phản xạ cực mạnh từ kênh **NIR/NDVI**).


* **Vùng đất đang giải phóng mặt bằng / Công trường san nền:**
* Các khu đất trống bị cạo lớp phủ thực vật để lộ nền cát/đất đỏ chuẩn bị xây dựng (kênh **BSI/SWIR** bắt tín hiệu đất trơ rực sáng).



---

### 2. Các đối tượng **DETECT DẠNG CỤM (Clustered Pattern), KHÔNG THỂ TÁCH RỜI ĐƠN LẺ**

Do 1 pixel rộng $\approx 20\text{m}$, những đối tượng này bị gộp chung vào một điểm ảnh (Mixed Pixel):

* **Nhà ở dân cư nông thôn hoặc nhà ống đô thị:**
* Một ngôi nhà ống $5\text{m} \times 20\text{m}$ ($100\text{m}^2$) sẽ bị gộp chung với sân, mái hiên, ngõ đi hoặc tán cây bên cạnh vào cùng 1 pixel.
* **Khả năng nhận diện:** Mô hình không đếm được "xã này có 1.500 ngôi nhà", nhưng **phát hiện được toàn bộ khu vực lõi dân cư tập trung** và đo được mật độ xây dựng trung bình ($\% \text{Built-up Density}$).


* **Đường ngõ xóm, đường nội bộ:**
* Đường làng, ngõ ngách rộng $2 - 5\text{m}$ không thể nhìn thấy trực tiếp thành đường kẻ, nhưng làm biến đổi giá trị phản xạ của pixel xung quanh (tạo ra hỗn hợp đất/bê tông xen kẽ).



---

### 3. Các đối tượng **HOÀN TOÀN KHÔNG THỂ DETECT ĐƯỢC** (Vượt quá giới hạn vật lý)

Tuyệt đối không sử dụng ảnh này cho các bài toán nhận diện đối tượng vi mô (như huấn luyện YOLO tìm vật thể):

* Xe ô tô, xe máy, container di chuyển trên đường.
* Từng cây bóng mát đơn lẻ trên vỉa hè.
* Cột điện, trạm biến áp nhỏ, biển hiệu quảng cáo.
* Con người, gia súc ngoài đồng ruộng.

---

### 4. Tóm lược tác vụ khả thi nhất trên ảnh này

Với ảnh đa phổ kích thước này, bài toán khả thi và mang lại giá trị cao nhất là **Phân vùng ngữ nghĩa (Semantic Segmentation)** bằng các mô hình như **U-Net, DeepLabV3+ hoặc SegFormer**. Bạn có thể chia bức ảnh thành **5 lớp đối tượng bề mặt (Masks)**:

1. *Lớp Đất công nghiệp/xây dựng lớn* (Nhà xưởng, kho, công trình kiên cố).
2. *Lớp Khu dân cư/đô thị nén* (Nhà ở tập trung, ngõ xóm).
3. *Lớp Đất nông nghiệp/cây xanh* (Đồng lúa, vườn, rừng).
4. *Lớp Mặt nước* (Sông, ao nuôi thủy sản).
5. *Lớp Đất trống/công trường xây dựng* (Đang san lấp).

Tỷ lệ diện tích và cấu trúc không gian của 5 lớp này chính là các đặc trưng định lượng mạnh nhất để đưa vào mô hình dự báo kinh tế - xã hội.